In [1]:
import os
from  typing_extensions import TypedDict


from dotenv import load_dotenv
from langchain_groq import  ChatGroq
from langchain_core.documents import  Document
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_huggingface import  HuggingFaceEmbeddings
from langchain_astradb import AstraDBVectorStore
from langchain_community.document_loaders import PyPDFLoader

from langgraph.graph import  StateGraph,END,START
from langgraph.prebuilt import ToolNode,tools_condition
from langgraph.graph.message import  MessagesState
from langchain_core.tools import  tool
from pydantic import BaseModel

from langchain_core.messages import  HumanMessage,SystemMessage
from langgraph.types import  Command ,interrupt
from langgraph.checkpoint.memory import  MemorySaver


C:\Users\Malik\AppData\Local\Temp\ipykernel_25220\534590624.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [22]:


load_dotenv()
# Set your Astra DB credentials
os.environ["ASTRA_DB_API_ENDPOINT"] = os.getenv("Astra_DB_API_Endpoint")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("Astra_DB_Token")
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")


In [2]:
emb=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:



vectore_store=AstraDBVectorStore(

embedding=emb,
collection_name="GENRAL_TASKS",

api_endpoint= os.getenv("Astra_DB_API_Endpoint"),
token=os.getenv("Astra_DB_Token")
)

In [7]:
'''
sample_texts = [
    "Astra DB is a serverless vector database built on Apache Cassandra.",
    "LangGraph allows you to build stateful, multi-actor applications with LLMs.",
    "RAG systems combine retrieval of private data with generative AI models to prevent hallucinations."
]
docs=[Document(page_content=text,metadata={"source":"notebook_test"}) for text in sample_texts]
'''
LOADER=PyPDFLoader("stilaite.pdf")
pdf_pages=LOADER.load()


text_spliter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits_docs=text_spliter.split_documents(pdf_pages)

splits_docs


[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:0d14211)', 'creationdate': '', 'author': 'Florian Cheyssial; Laurent M. Mugnier; Cyril Petit', 'doi': 'https://doi.org/10.48550/arXiv.2609.17023', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Optimal exposure time for satellite imaging? A trade-off between residual tip-tilt and registration accuracy', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2609.17023v1', 'source': 'stilaite.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='To be published inAdaptive Optics Systems X, volume 14150, SPIE Astronomical Telescopes +\nInstrumentation, Copenhagen, 2026.\nOptimal exposure time for satellite imaging? A trade-off\nbetween residual tip-tilt and registration accuracy\nFlorian Cheyssiala, Laurent M. Mugnier a, and Cyril Petit a\naDOTA, ONERA, U

In [9]:
print(f"Loaded {len(pdf_pages)} pages from the PDF.")
print(f"Split the PDF into {len(splits_docs)} chunks.")


Loaded 7 pages from the PDF.
Split the PDF into 36 chunks.


In [10]:
insert_ids=vectore_store.add_documents(splits_docs)

In [13]:
print(f"Inserted {len(insert_ids)} document chunks into Astra DB.")

Inserted 3 document chunks into Astra DB.


In [12]:
query = "TRADE-OFF BETWEEN INTEGRATED JITTER & REGISTRATION ERROR"


res=vectore_store.similarity_search_with_score(query,k=3)
for doc ,score in res:
    print(f"score :{score} the doc:{doc}")


    


score :0.89365655 the doc:page_content='certain threshold (here around 10 ms), all the jitter statistic is integrated. Past this threshold, there is no interest
in making registration. Compared to a long pose, jitter effects are reduced by≈66% at the optimal exposure
time.
To make a comparison with the CRLB, we calculate empirically the standard deviation of the registration
error,σ reg, for two cases. In the first case, the reference imagegis known, as it is supposed in the calculation of
5 of 7' metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:0d14211)', 'creationdate': '', 'author': 'Florian Cheyssial; Laurent M. Mugnier; Cyril Petit', 'doi': 'https://doi.org/10.48550/arXiv.2609.17023', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Optimal exposure time for satellite imaging? A trade-off between residual tip-tilt and regi

In [6]:
retriver=vectore_store.as_retriever(search_kwargs={"k":2})







In [ ]:
ress=retriver.invoke("who was the author? ")
for  i , doc in enumerate(ress):
    res=llm.invoke(doc.page_content)
    print(res.content)

It looks like you’ve pasted a short list of references (items 5‑7) that relate to pyramid wave‑front sensors and their optical gains. How can I help you with them?  

Some common requests include:

* **Formatting the citations** in a specific style (APA, IEEE, Chicago, etc.).  
* **Creating a complete bibliography** (adding missing details such as DOIs, page ranges, or publisher information).  
* **Summarizing the key findings** of each paper.  
* **Explaining how to incorporate the “pyramid optical gains”** from the Fetick et al. (2023) conference paper into an analytical model.  
* **Generating in‑text citations** for a manuscript you’re writing.  

Just let me know which of the above (or anything else) you need, and I’ll take it from there!
Below is a compact yet complete walkthrough of the material you quoted, together with practical pointers on how to turn the theory into an implementable algorithm. I’ve split the answer into three logical blocks:

1. **What “tip‑tilt PSD” means a

In [24]:
# SCHEMA OF THE 


class AgentState(MessagesState):
    query:str
    context:str
    draft:str
    feedback:str
    final:str

In [ ]:
# nodes 

def retriver_node(state:MessagesState):

    return ""

def draft_node(state:MessagesState):
    return 


def humman_revivew_node(state:MessagesState):
    return ""

def finalize_node(state:MessagesState):
    return ""




    


In [ ]:
# graphbuilding

graph=StateGraph(MessagesState)

graph.add_node()
